# BERT Reranker on PyTorch Native + Triton Inference Server (trn2, LNC=2)

Deploys [Alibaba-NLP/gte-multilingual-reranker-base](https://huggingface.co/Alibaba-NLP/gte-multilingual-reranker-base)
behind NVIDIA Triton Inference Server using **PyTorch Native**
(`torch.compile(model, backend="neuron")`), the supported compile path going forward.
The earlier notebooks in this directory use `torch_neuronx.trace()`, which is deprecated and
has no environment in SDK 2.32.

**Instance**: trn2.3xlarge, LNC=2, **4 Triton model instances** (one per logical core)
**Container**: PyTorch Native Beta 4 (torch 2.11.0, torch-neuronx 2.11.3, neuronx-cc 2.26, nki 0.5.0)
**dtype**: full BF16

## How this differs from the trace notebooks

| | trace notebooks | this notebook |
|---|---|---|
| compile API | `torch_neuronx.trace()` -> `.pt` | `torch.compile(backend="neuron")` |
| artifact shipped to Triton | traced TorchScript `.pt` per batch size | persistent **NEFF cache directory** |
| what each instance does at start | `torch.jit.load` | build model, `torch.compile`, warm -> **cache hit** |
| base image | SDK 2.31 `pytorch-inference-neuronx` | PyTorch Native Beta 4 |

The NEFF cache is the important part. Without it every instance would recompile every bucket
at startup (~1.5-2 min each). The cache is populated once, ahead of time, and mounted into the
container read-only via `TORCH_NEURONX_NEFF_CACHE_DIR`.


---
## Step 0: Environment

Everything that touches the Neuron device runs in a **subprocess or container**, never in this
kernel: a process that initialises the Neuron runtime keeps its cores until it exits, which
would starve the Triton container later.


In [1]:
import os, subprocess, sys, time, json, shutil

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)
    return r.stdout.strip(), r.stderr.strip(), r.returncode

PLATFORM = "trn2"
LNC = "2"
NUM_INSTANCES = 4
BETA4 = "421672808698.dkr.ecr.us-east-1.amazonaws.com/concourse-release-0461d3b:2.11.0-neuronx-py312-sdk2.31.0-ubuntu24.04-neurondlcbuilder-development-6488856940-0"
MODEL_ID = "Alibaba-NLP/gte-multilingual-reranker-base"
SEQ_LENGTH = 1024
BATCH_SIZES = [1, 2, 4, 8, 16]
HOME = os.path.expanduser("~")
WORK = f"{HOME}/native_triton"
CACHE = f"{WORK}/neff_cache"          # persistent NEFF cache, shared by all instances
REPO = f"{WORK}/model_repo"
os.makedirs(CACHE, exist_ok=True)

out, _, _ = sh(f"NEURON_LOGICAL_NC_CONFIG={LNC} neuron-ls")
print(out)
out, _, _ = sh(["docker", "images", "-q", BETA4])
print("Beta 4 image present:", bool(out))
if not out:
    raise SystemExit("pull the Beta 4 image first (ECR login to 421672808698)")
try:
    import tritonclient.http  # noqa
except ImportError:
    sh(f"{sys.executable} -m pip install -q tritonclient[http]")
out, _, _ = sh("free -g | awk '/Swap/ {print $2}'")
print("swap GB:", out)

instance-type: trn2.3xlarge
instance-id: i-<redacted>
logical-neuroncore-config: 2
+--------+--------+----------+--------+--------------+----------+------+
| NEURON | NEURON |  NEURON  | NEURON |     PCI      |   CPU    | NUMA |
| DEVICE | CORES  | CORE IDS | MEMORY |     BDF      | AFFINITY | NODE |
+--------+--------+----------+--------+--------------+----------+------+
| 0      | 4      | 0-3      | 96 GB  | 0000:33:00.0 | 0-11     | 0    |
+--------+--------+----------+--------+--------------+----------+------+
Beta 4 image present: True
swap GB: 63


---
## Step 1: Populate the NEFF cache (one compile per batch size)

Runs inside the Beta 4 container on **one** core. Each bucket is compiled with
`torch.compile(..., backend="neuron", dynamic=False)` and written to the shared cache. Accuracy
is checked against a CPU FP32 reference in the same run.


In [2]:
precompile = r"""
import os, sys, time, json
import numpy as np, torch, torch_neuronx
from transformers import AutoModelForSequenceClassification, AutoTokenizer
M = "Alibaba-NLP/gte-multilingual-reranker-base"; SEQ = 1024
BUCKETS = [1, 2, 4, 8, 16]
dev = torch.device("neuron")
tok = AutoTokenizer.from_pretrained(M, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(
    M, trust_remote_code=True, attn_implementation="eager").eval()
model = model.to(torch.bfloat16).to(dev)        # .to(device) BEFORE compile

compiled = {}
for bs in BUCKETS:
    cm = torch.compile(model, backend="neuron", dynamic=False)
    ids = torch.zeros((bs, SEQ), dtype=torch.long, device=dev)
    mask = torch.ones((bs, SEQ), dtype=torch.long, device=dev)
    t0 = time.time()
    with torch.no_grad():
        cm(ids, mask); torch_neuronx.synchronize()
    print(f"  BS={bs:>2}: compiled + first run in {time.time()-t0:.0f}s", flush=True)
    compiled[bs] = cm

# accuracy vs CPU FP32
Q = "What are the benefits of renewable energy?"
P = ["Renewable energy sources like solar and wind power produce electricity without "
     "greenhouse gas emissions, reducing climate change impacts.",
     "The stock market experienced significant volatility in Q4, with tech stocks leading "
     "the decline amid rising interest rates.",
     "Solar panels have become 90% cheaper over the past decade, making renewable energy "
     "cost-competitive with fossil fuels.",
     "The history of ancient Rome spans over a thousand years, from its founding in 753 BC.",
     "Wind energy creates more jobs per megawatt than coal or natural gas power plants.",
     "Python is a popular programming language known for its readability."]
e = tok([Q]*len(P), P, return_tensors="pt", max_length=SEQ, padding="max_length", truncation=True)
cpu = AutoModelForSequenceClassification.from_pretrained(
    M, trust_remote_code=True, attn_implementation="eager").eval()
with torch.no_grad():
    ref = cpu(e["input_ids"], e["attention_mask"]).logits[:, 0].float().numpy()
pad = 16 - len(P)
ai = torch.cat([e["input_ids"], e["input_ids"][:1].repeat(pad, 1)]).to(dev)
am = torch.cat([e["attention_mask"], e["attention_mask"][:1].repeat(pad, 1)]).to(dev)
with torch.no_grad():
    sc = compiled[16](ai, am).logits[:len(P), 0].float().cpu().numpy()
cos = float(np.dot(sc, ref) / (np.linalg.norm(sc) * np.linalg.norm(ref)))
top3 = sorted(np.argsort(-sc)[:3].tolist())
print(f"cosine vs CPU FP32: {cos:.6f}   top-3: {top3}")
assert cos > 0.999 and top3 == [0, 2, 4], "accuracy gate failed"
print("ACCURACY GATE PASSED")
cache = os.environ["TORCH_NEURONX_NEFF_CACHE_DIR"]
n = sum(len(f) for _, _, f in os.walk(cache))
print(f"NEFF cache: {cache}  ({n} files)")
"""
with open(f"{WORK}/precompile.py", "w") as f:
    f.write(precompile)
with open(f"{WORK}/precompile.sh", "w") as f:
    f.write("pip install -q 'transformers==4.53.3' 2>&1 | tail -1\npython3 /work/precompile.py\n")

cmd = ["sudo", "docker", "run", "--rm", "--entrypoint", "bash",
       "--device", "/dev/neuron0",
       "-e", f"NEURON_LOGICAL_NC_CONFIG={LNC}",
       "-e", f"NEURON_CC_FLAGS=--lnc={LNC}",
       "-e", "NEURON_RT_VISIBLE_CORES=0",
       "-e", "NEURON_RT_LOG_LEVEL=ERROR",
       "-e", "TORCH_NEURONX_NEFF_CACHE_DIR=/cache",
       "-v", f"{WORK}:/work", "-v", f"{CACHE}:/cache",
       "-v", f"{HOME}/.cache/huggingface:/root/.cache/huggingface",
       BETA4, "/work/precompile.sh"]
t0 = time.time()
r = subprocess.run(cmd, capture_output=True, text=True, timeout=7200)
keep = [l for l in r.stdout.splitlines() + r.stderr.splitlines()
        if any(k in l for k in ("BS=", "cosine", "GATE", "NEFF cache", "Error", "error:"))]
print("\n".join(keep))
print(f"\nprecompile took {(time.time()-t0)/60:.1f} min")
if "ACCURACY GATE PASSED" not in r.stdout:
    print(r.stderr[-3000:])
    raise SystemExit("precompile failed")

  BS= 1: compiled + first run in 46s
  BS= 2: compiled + first run in 34s
  BS= 4: compiled + first run in 48s
  BS= 8: compiled + first run in 41s
  BS=16: compiled + first run in 48s
cosine vs CPU FP32: 0.999982   top-3: [0, 2, 4]
ACCURACY GATE PASSED
NEFF cache: /cache  (40 files)

precompile took 4.4 min


---
## Step 2: Triton model repository

`model.py` builds the model in `initialize()`, pins itself to one core with
`NEURON_RT_VISIBLE_CORES` (set **before** `torch_neuronx` is imported), compiles each bucket
(cache hit), and warms it. `execute()` routes each request to the smallest bucket that fits.


In [3]:
os.makedirs(f"{REPO}/bert_reranker/1", exist_ok=True)
config = f"""name: "bert_reranker"
backend: "python"
max_batch_size: 16
input [
  {{ name: "input_ids"      data_type: TYPE_INT64 dims: [ {SEQ_LENGTH} ] }},
  {{ name: "attention_mask" data_type: TYPE_INT64 dims: [ {SEQ_LENGTH} ] }}
]
output [ {{ name: "score" data_type: TYPE_FP32 dims: [ 1 ] }} ]
instance_group [ {{ kind: KIND_MODEL count: {NUM_INSTANCES} }} ]
dynamic_batching {{
  preferred_batch_size: [ {', '.join(map(str, BATCH_SIZES))} ]
  max_queue_delay_microseconds: 5000
}}
parameters [
  {{ key: "SEQ_LENGTH"    value: {{ string_value: "{SEQ_LENGTH}" }} }},
  {{ key: "BATCH_SIZES"   value: {{ string_value: "{','.join(map(str, BATCH_SIZES))}" }} }},
  {{ key: "NUM_INSTANCES" value: {{ string_value: "{NUM_INSTANCES}" }} }},
  {{ key: "STAGGER_SEC"   value: {{ string_value: "3" }} }}
]
"""
open(f"{REPO}/bert_reranker/config.pbtxt", "w").write(config)

model_py = r"""
import json, os, sys, time
import numpy as np
try:
    import triton_python_backend_utils as pb_utils
except ImportError:
    pb_utils = None

MODEL_ID = "Alibaba-NLP/gte-multilingual-reranker-base"


class TritonPythonModel:
    def initialize(self, args):
        cfg = json.loads(args["model_config"])
        p = lambda k, d: cfg.get("parameters", {}).get(k, {}).get("string_value", d)
        self.seq = int(p("SEQ_LENGTH", "1024"))
        self.buckets = sorted(int(b) for b in p("BATCH_SIZES", "1,2,4,8,16").split(","))
        n_inst = int(p("NUM_INSTANCES", "1"))
        stagger = float(p("STAGGER_SEC", "3"))
        idx = int(args.get("model_instance_name", "x_0").rsplit("_", 1)[-1] or 0)
        core = idx % n_inst

        # Must be set before torch_neuronx initialises the runtime in this process.
        os.environ["NEURON_RT_VISIBLE_CORES"] = str(core)
        if idx and stagger:
            time.sleep(stagger * idx)

        import torch, torch_neuronx
        from transformers import AutoModelForSequenceClassification
        self.torch, self.tnx = torch, torch_neuronx
        self.dev = torch.device("neuron")
        m = AutoModelForSequenceClassification.from_pretrained(
            MODEL_ID, trust_remote_code=True, attn_implementation="eager").eval()
        m = m.to(torch.bfloat16).to(self.dev)
        self.models = {}
        t0 = time.time()
        for bs in self.buckets:
            cm = torch.compile(m, backend="neuron", dynamic=False)
            ids = torch.zeros((bs, self.seq), dtype=torch.long, device=self.dev)
            mask = torch.ones((bs, self.seq), dtype=torch.long, device=self.dev)
            with torch.no_grad():
                for _ in range(3):
                    cm(ids, mask)
                torch_neuronx.synchronize()
            self.models[bs] = cm
        self._log(f"instance {idx} -> core {core}; buckets {self.buckets} ready "
                  f"in {time.time()-t0:.0f}s")

    def _log(self, msg):
        (pb_utils.Logger.log_info if pb_utils else print)(f"[bert_reranker] {msg}")

    def _route(self, n):
        for b in self.buckets:
            if b >= n:
                return b
        return self.buckets[-1]

    def execute(self, requests):
        torch = self.torch
        out = []
        for req in requests:
            ids_np = pb_utils.get_input_tensor_by_name(req, "input_ids").as_numpy()
            mask_np = pb_utils.get_input_tensor_by_name(req, "attention_mask").as_numpy()
            n = ids_np.shape[0]
            bs = self._route(n)
            ids = torch.as_tensor(ids_np, dtype=torch.long)
            mask = torch.as_tensor(mask_np, dtype=torch.long)
            if n < bs:
                ids = torch.cat([ids, ids[:1].repeat(bs - n, 1)])
                mask = torch.cat([mask, mask[:1].repeat(bs - n, 1)])
            elif n > bs:
                ids, mask = ids[:bs], mask[:bs]
            with torch.no_grad():
                logits = self.models[bs](ids.to(self.dev), mask.to(self.dev)).logits
            scores = logits[:n, 0:1].float().cpu().numpy().astype(np.float32)
            out.append(pb_utils.InferenceResponse(
                output_tensors=[pb_utils.Tensor("score", scores)]))
        return out

    def finalize(self):
        self._log("finalize")
"""
open(f"{REPO}/bert_reranker/1/model.py", "w").write(model_py)
print(open(f"{REPO}/bert_reranker/config.pbtxt").read())

name: "bert_reranker"
backend: "python"
max_batch_size: 16
input [
  { name: "input_ids"      data_type: TYPE_INT64 dims: [ 1024 ] },
  { name: "attention_mask" data_type: TYPE_INT64 dims: [ 1024 ] }
]
output [ { name: "score" data_type: TYPE_FP32 dims: [ 1 ] } ]
instance_group [ { kind: KIND_MODEL count: 4 } ]
dynamic_batching {
  preferred_batch_size: [ 1, 2, 4, 8, 16 ]
  max_queue_delay_microseconds: 5000
}
parameters [
  { key: "SEQ_LENGTH"    value: { string_value: "1024" } },
  { key: "BATCH_SIZES"   value: { string_value: "1,2,4,8,16" } },
  { key: "NUM_INSTANCES" value: { string_value: "4" } },
  { key: "STAGGER_SEC"   value: { string_value: "3" } }
]



---
## Step 3: Standalone backend test (mocked `pb_utils`, inside the container)

Runs `model.py`'s full lifecycle without `tritonserver`: routing, **padding isolation**
(bit-exact), cross-bucket agreement, and **ranking stability across buckets**. Uses the NEFF
cache from Step 1, so it also proves the cache is hit.


In [4]:
test_py = r"""
import json, sys, types, time
import numpy as np
class T:
    def __init__(s, n, a): s.n, s.a = n, a
    def as_numpy(s): return s.a
class R:
    def __init__(s, output_tensors): s.o = output_tensors
sys.modules["triton_python_backend_utils"] = types.SimpleNamespace(
    Logger=types.SimpleNamespace(log_info=print), Tensor=T, InferenceResponse=R,
    get_input_tensor_by_name=lambda r, n: r[n])
sys.path.insert(0, "/models/bert_reranker/1")
from model import TritonPythonModel
from transformers import AutoTokenizer
t0 = time.time()
m = TritonPythonModel()
m.initialize({"model_config": json.dumps({"parameters": {
    "SEQ_LENGTH": {"string_value": "1024"},
    "BATCH_SIZES": {"string_value": "1,2,4,8,16"},
    "NUM_INSTANCES": {"string_value": "1"}, "STAGGER_SEC": {"string_value": "0"}}}),
    "model_instance_name": "bert_reranker_0"})
print(f"initialize took {time.time()-t0:.0f}s (cache hit expected: well under a minute per bucket)")
tok = AutoTokenizer.from_pretrained("Alibaba-NLP/gte-multilingual-reranker-base",
                                    trust_remote_code=True)
Q = "What are the benefits of renewable energy?"
P = ["Solar and wind power produce electricity without greenhouse gas emissions.",
     "The stock market experienced volatility in Q4 amid rising rates.",
     "Solar panels are 90% cheaper than a decade ago."]
FILL = "Completely unrelated text about ancient Roman architecture and history."
def run(qs, ds):
    e = tok(qs, ds, return_tensors="np", max_length=1024, padding="max_length", truncation=True)
    req = {"input_ids": T("i", e["input_ids"].astype(np.int64)),
           "attention_mask": T("m", e["attention_mask"].astype(np.int64))}
    return m.execute([req])[0].o[0].a
for n in [1, 3, 5, 12, 16]:
    assert run([Q]*n, (P*8)[:n]).shape == (n, 1)
print("routing OK")
for n in [2, 4, 8, 16]:
    a = float(run([Q]*n, [P[0]]*n)[0, 0]); b = float(run([Q]*n, [P[0]] + [FILL]*(n-1))[0, 0])
    assert a == b, f"padding leak at n={n}: {a} vs {b}"
print("padding isolation: bit-exact at n=2,4,8,16")
base = float(run([Q], [P[0]])[0, 0])
for n in [2, 4, 8, 16]:
    d = abs(float(run([Q]*n, [P[0]]*n)[0, 0]) - base)
    assert d < 0.1, f"cross-bucket delta {d}"
print("cross-bucket agreement OK")
orders = set()
for n in [4, 8, 16]:
    s = run([Q]*n, P + [FILL]*(n-3))[:3, 0]
    orders.add(tuple(np.argsort(-s).tolist()))
assert len(orders) == 1, f"ranking changed across buckets: {orders}"
print(f"ranking stable across buckets: {orders.pop()}")
print("STANDALONE TEST PASSED")
"""
open(f"{WORK}/standalone_test.py", "w").write(test_py)
open(f"{WORK}/standalone.sh", "w").write(
    "pip install -q 'transformers==4.53.3' 2>&1 | tail -1\npython3 /work/standalone_test.py\n")
cmd = ["sudo", "docker", "run", "--rm", "--entrypoint", "bash", "--device", "/dev/neuron0",
       "-e", f"NEURON_LOGICAL_NC_CONFIG={LNC}", "-e", f"NEURON_CC_FLAGS=--lnc={LNC}",
       "-e", "NEURON_RT_LOG_LEVEL=ERROR", "-e", "TORCH_NEURONX_NEFF_CACHE_DIR=/cache",
       "-v", f"{WORK}:/work", "-v", f"{CACHE}:/cache", "-v", f"{REPO}:/models",
       "-v", f"{HOME}/.cache/huggingface:/root/.cache/huggingface",
       BETA4, "/work/standalone.sh"]
r = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
keep = [l for l in r.stdout.splitlines() if not l.startswith(("[W", "  ", "W1", "I1"))]
print("\n".join(keep[-14:]))
if "STANDALONE TEST PASSED" not in r.stdout:
    print(r.stderr[-3000:])
    raise SystemExit("standalone test failed")

[notice] To update, run: pip install --upgrade pip
[bert_reranker] instance 0 -> core 0; buckets [1, 2, 4, 8, 16] ready in 37s
initialize took 44s (cache hit expected: well under a minute per bucket)
routing OK
padding isolation: bit-exact at n=2,4,8,16
cross-bucket agreement OK
ranking stable across buckets: (0, 2, 1)
STANDALONE TEST PASSED


---
## Step 4: Build Triton on top of the Beta 4 container

Triton r26.01 (python backend only) is built from source against the container's python, so the
backend stub runs the same interpreter that has `torch_neuronx` and the `neuron` dynamo backend.


In [5]:
IMAGE = "triton-native-beta4:r26.01"
out, _, _ = sh(["sudo", "docker", "images", "-q", IMAGE])
if out:
    print("image exists:", out)
else:
    dockerfile = f"""FROM {BETA4}
ENV DEBIAN_FRONTEND=noninteractive
# The Beta 4 image ships an apt source for an internal beta repo that returns 401 Unauthorized,
# which makes every apt-get update fail. Drop any such source before installing build deps.
RUN grep -rl "artifacts.beta.neuron" /etc/apt/sources.list /etc/apt/sources.list.d/ 2>/dev/null \\
      | xargs -r rm -f \\
 && apt-get update && apt-get install -y --no-install-recommends \\
      git build-essential pkg-config rapidjson-dev libb64-dev libre2-dev libssl-dev \\
      libcurl4-openssl-dev libnuma-dev libarchive-dev zlib1g-dev libboost-dev \\
      autoconf automake libtool patchelf && rm -rf /var/lib/apt/lists/*
RUN python3 -m pip install --no-cache-dir "cmake==3.31.10" wheel setuptools distro requests build virtualenv \\
    && python3 -m pip install --no-cache-dir "transformers==4.53.3"
RUN PY=$(python3 -c 'import sys;print(sys.executable)') \\
 && PYINC=$(python3 -c 'import sysconfig;print(sysconfig.get_paths()["include"])') \\
 && PYLIB=$(python3 -c 'import sysconfig,os;print(os.path.join(sysconfig.get_config_var("LIBDIR"),sysconfig.get_config_var("LDLIBRARY")))') \\
 && echo "python=$PY include=$PYINC lib=$PYLIB" \\
 && git clone --depth=1 --branch=r26.01 https://github.com/triton-inference-server/server.git /server \\
 && cd /server \\
 && Python3_EXECUTABLE=$PY Python3_INCLUDE_DIR=$PYINC Python3_LIBRARY=$PYLIB \\
    ./build.py -v --no-container-build --build-dir=/server/build --backend=python \\
      --enable-logging --enable-stats --endpoint=http --endpoint=grpc \\
 && cp -r /server/build/opt/* /opt/ && cd / && rm -rf /server
ENV PATH=/opt/tritonserver/bin:$PATH
ENTRYPOINT []
"""
    open(f"{WORK}/Dockerfile", "w").write(dockerfile)
    t0 = time.time()
    r = subprocess.run(["sudo", "docker", "build", "-t", IMAGE, "-f", f"{WORK}/Dockerfile", WORK],
                       capture_output=True, text=True, timeout=5400)
    print(f"build {'OK' if r.returncode == 0 else 'FAILED'} in {(time.time()-t0)/60:.1f} min")
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise SystemExit("docker build failed")

image exists: fbefd4d08390


---
## Step 5: Start Triton

The NEFF cache and the HuggingFace cache are mounted into the container, so instance start is a
cache hit. Readiness is polled on `/v2/models/{name}/ready`.


In [6]:
import requests
CONTAINER = "triton-native"
sh(["sudo", "docker", "rm", "-f", CONTAINER])
cmd = ["sudo", "docker", "run", "-d", "--name", CONTAINER, "--device", "/dev/neuron0",
       "--shm-size=8g", "-p", "8000:8000", "-p", "8001:8001", "-p", "8002:8002",
       "-e", f"NEURON_LOGICAL_NC_CONFIG={LNC}", "-e", f"NEURON_CC_FLAGS=--lnc={LNC}",
       "-e", "NEURON_RT_LOG_LEVEL=ERROR", "-e", "TORCH_NEURONX_NEFF_CACHE_DIR=/cache",
       "-e", "TOKENIZERS_PARALLELISM=false",
       "-v", f"{REPO}:/models", "-v", f"{CACHE}:/cache",
       "-v", f"{HOME}/.cache/huggingface:/root/.cache/huggingface",
       IMAGE, "tritonserver", "--model-repository=/models", "--exit-on-error=true"]
print(sh(cmd)[0][:16])
t0 = time.time(); ready = False
while time.time() - t0 < 3600:
    try:
        if requests.get("http://localhost:8000/v2/models/bert_reranker/ready", timeout=2).status_code == 200:
            ready = True; break
    except Exception:
        pass
    time.sleep(10)
logs, _, _ = sh(f"sudo docker logs {CONTAINER} 2>&1 | grep -E 'buckets|Error|error' | tail -20")
print(logs)
if not ready:
    print(sh(f"sudo docker logs --tail 80 {CONTAINER} 2>&1")[0])
    raise SystemExit("server not ready")
print(f"SERVER READY in {(time.time()-t0)/60:.1f} min")

0ea09b0b069ebaf7


I1005 11:31:59.616699 1 model.py:50] "[bert_reranker] instance 0 -> core 0; buckets [1, 2, 4, 8, 16] ready in 38s"
I1005 11:32:09.038209 1 model.py:50] "[bert_reranker] instance 1 -> core 1; buckets [1, 2, 4, 8, 16] ready in 37s"
I1005 11:32:21.358794 1 model.py:50] "[bert_reranker] instance 2 -> core 2; buckets [1, 2, 4, 8, 16] ready in 38s"
I1005 11:32:21.402165 1 model.py:50] "[bert_reranker] instance 3 -> core 3; buckets [1, 2, 4, 8, 16] ready in 38s"
SERVER READY in 1.3 min


---
## Step 6: Benchmark

Same methodology as the trace notebooks: threaded HTTP workers, 10 s per configuration,
5-request warmup, throughput and P50/P95/P99.


In [7]:
import threading, numpy as np
import tritonclient.http as httpclient
from queue import Queue
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

def make(bs):
    e = tok(["What is machine learning?"] * bs,
            ["Machine learning is a subset of artificial intelligence that focuses on "
             "building systems that learn from data."] * bs,
            max_length=SEQ_LENGTH, padding="max_length", truncation=True, return_tensors="np")
    a = httpclient.InferInput("input_ids", e["input_ids"].shape, "INT64")
    b = httpclient.InferInput("attention_mask", e["attention_mask"].shape, "INT64")
    a.set_data_from_numpy(e["input_ids"].astype(np.int64))
    b.set_data_from_numpy(e["attention_mask"].astype(np.int64))
    return [a, b]

# tritonclient.http is gevent-based: a client object is bound to the thread that first uses it.
# Each worker therefore creates and uses its OWN client inside its own thread, and warmup uses a
# separate client. (Reusing one client across threads raises greenlet.error and silently kills
# that worker.) Every worker's sample count is checked so a dead worker cannot go unnoticed.
def run(bs, nw, dur=10.0):
    inp = make(bs)
    warm = httpclient.InferenceServerClient(url="localhost:8000")
    for _ in range(5):
        warm.infer("bert_reranker", inp)
    q, counts, errs = Queue(), [0] * nw, []
    def w(k):
        c = httpclient.InferenceServerClient(url="localhost:8000")
        end = time.time() + dur
        try:
            while time.time() < end:
                s = time.time(); c.infer("bert_reranker", inp)
                q.put((time.time() - s) * 1000); counts[k] += 1
        except Exception as e:
            errs.append(f"worker {k}: {type(e).__name__}: {e}")
    ts = [threading.Thread(target=w, args=(k,)) for k in range(nw)]
    t0 = time.time(); [t.start() for t in ts]; [t.join() for t in ts]
    el = time.time() - t0
    if errs or min(counts) == 0:
        raise RuntimeError(f"BS={bs} W={nw}: {sum(c == 0 for c in counts)} idle workers; {errs[:3]}")
    lat = [q.get() for _ in range(q.qsize())]
    return {"batch": bs, "workers": nw, "active_workers": sum(c > 0 for c in counts),
            "requests": len(lat),
            "p50": float(np.percentile(lat, 50)), "p95": float(np.percentile(lat, 95)),
            "p99": float(np.percentile(lat, 99)), "throughput": len(lat) * bs / el}

rows = []
print(f"{'BS':>3} {'W':>4} {'p50':>9} {'p95':>9} {'p99':>9} {'inf/s':>9}")
for nw in ([1, 8, 16, 32] if PLATFORM == "inf2" else [1, 16, 32, 64]):
    for bs in BATCH_SIZES:
        r = run(bs, nw); rows.append(r)
        print(f"{bs:>3} {nw:>4} {r['p50']:>9.2f} {r['p95']:>9.2f} {r['p99']:>9.2f} {r['throughput']:>9.1f}")
json.dump(rows, open(f"{HOME}/triton_bench_native_{PLATFORM}_lnc{LNC}.json", "w"), indent=2)
best = max(rows, key=lambda r: r["throughput"])
u = [r for r in rows if r["p50"] < 100]
print(f"\nPEAK: {best['throughput']:.1f} inf/s at BS={best['batch']}, {best['workers']} workers "
      f"(p50 {best['p50']:.2f} ms)")
if u:
    b = max(u, key=lambda r: r["throughput"])
    print(f"best under 100 ms p50: {b['throughput']:.1f} inf/s at BS={b['batch']}, "
          f"{b['workers']} workers (p50 {b['p50']:.2f} ms)")

 BS    W       p50       p95       p99     inf/s


  1    1     13.30     14.19     14.53      74.8


  2    1     22.29     23.13     23.43      89.4


  4    1     42.88     43.66     43.96      93.1


  8    1     76.30     77.44     77.60     104.7


...

  1   64    203.51    236.40    259.60     339.1


  2   64    326.99    329.65    331.25     389.7


  4   64    656.32    659.08    661.15     389.7


  8   64   1202.45   1209.29   1213.92     425.2


 16   64   2419.63   2424.16   2424.73     423.0

PEAK: 425.8 inf/s at BS=8, 16 workers (p50 300.36 ms)
best under 100 ms p50: 389.7 inf/s at BS=2, 16 workers (p50 85.29 ms)


---
## Step 7: Sample reranking and cleanup


In [8]:
Q = "What are the benefits of renewable energy?"
P = ["Renewable energy sources like solar and wind power produce electricity without "
     "greenhouse gas emissions, reducing climate change impacts.",
     "The stock market experienced significant volatility in Q4, with tech stocks leading "
     "the decline amid rising interest rates.",
     "Solar panels have become 90% cheaper over the past decade, making renewable energy "
     "cost-competitive with fossil fuels.",
     "The history of ancient Rome spans over a thousand years, from its founding in 753 BC.",
     "Wind energy creates more jobs per megawatt than coal or natural gas power plants.",
     "Python is a popular programming language known for its readability."]
c = httpclient.InferenceServerClient(url="localhost:8000")
scored = []
for i, p in enumerate(P):
    e = tok([Q], [p], max_length=SEQ_LENGTH, padding="max_length", truncation=True, return_tensors="np")
    a = httpclient.InferInput("input_ids", e["input_ids"].shape, "INT64")
    b = httpclient.InferInput("attention_mask", e["attention_mask"].shape, "INT64")
    a.set_data_from_numpy(e["input_ids"].astype(np.int64)); b.set_data_from_numpy(e["attention_mask"].astype(np.int64))
    scored.append((float(c.infer("bert_reranker", [a, b]).as_numpy("score")[0][0]), i, p))
scored.sort(key=lambda x: -x[0])
for rank, (s, i, p) in enumerate(scored, 1):
    print(f"{rank}  {s:+.3f}  [{i}] {p[:70]}")
top3 = sorted(i for _, i, _ in scored[:3])
print("top-3 == relevant [0, 2, 4]:", top3 == [0, 2, 4])
sh(["sudo", "docker", "rm", "-f", CONTAINER])
print("container removed; NEFF cache kept at", CACHE)

1  +1.227  [0] Renewable energy sources like solar and wind power produce electricity
2  +0.910  [4] Wind energy creates more jobs per megawatt than coal or natural gas po
3  +0.902  [2] Solar panels have become 90% cheaper over the past decade, making rene
4  -3.312  [1] The stock market experienced significant volatility in Q4, with tech s
5  -3.328  [3] The history of ancient Rome spans over a thousand years, from its foun
6  -3.328  [5] Python is a popular programming language known for its readability.
top-3 == relevant [0, 2, 4]: True


container removed; NEFF cache kept at /home/ubuntu/native_triton/neff_cache
